# Diffusion and Flow Matching


Lecture 17 built the variational autoencoder: a one-step encoder to a simple
latent, a decoder back, and an ELBO to train both. This lecture replaces the
one learned step with a thousand fixed ones. A diffusion model's encoder is not
trained at all; it is a stochastic process that slowly destroys an image into
Gaussian noise, and the model learns only to run that process backwards. The
first half of the lecture derives the forward process, the reverse process and
the noise-prediction objective of Ho et al. (2020), following Prince,
*[Understanding Deep Learning](https://udlbook.github.io/udlbook/)* (UDL)
chapter 18 and
[Rocca (2022)](https://medium.com/data-science/understanding-diffusion-probabilistic-models-dpms-1940329d6048),
whose figures we reuse. The second half is what changed between 2021 and 2026:
the score function and the SDE view that unify everything, flow matching and
rectified flow (the formulation Stable Diffusion 3, Flux and the video models
use), and the engineering that makes a text-to-image model practical:
classifier-free guidance, latent diffusion, the diffusion transformer, few-step
distillation, and discrete diffusion for text.

The code cells run every idea exactly on a one-dimensional toy problem, a
two-component Gaussian mixture, whose noised marginals, score and flow velocity
all have closed forms, so nothing is trained. Code cells adapted from the UDL
notebooks (MIT license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).

Notation. The diffusion literature writes the noisy variables as
$x_0, x_1, \ldots, x_T$ with $x_0$ the data, the noise schedule as $\beta_t$
with $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \prod_{s \le t}\alpha_s$, and
the network parameters as $\theta$. UDL chapter 18 writes the same objects as
$\mathbf{z}_t$, $\beta_t$, and $\alpha_t$ for what the papers call
$\bar\alpha_t$, with parameters $\boldsymbol{\phi}$. We follow the papers so
that every equation can be looked up at its source. Flow matching runs time the
other way, from noise at $t = 0$ to data at $t = 1$; the lecture says so again
when it gets there. The quizzable derivations are the diffusion kernel
$q(x_t \mid x_0)$ (section 3), the relation $\epsilon = -\sigma_t s$ between
noise and score (section 6), and the argument that conditional flow matching has
the same gradient as flow matching (section 6).


## Today's roadmap

1. Contextualizing diffusion models
2. Diffusion processes: Markov, Wiener, SDEs and time reversal
3. The forward process and the diffusion kernel $q(x_t \mid x_0)$
4. The reverse process and the noise-prediction objective
5. Training, sampling and the U-Net
6. Score matching and flow matching: one model, three languages
7. Making it practical: guidance, latents, DiT, distillation, text
8. Applications
9. Summary


# Contextualizing diffusion models

We start by placing diffusion models next to the other generative models we
have seen (VAEs and GANs in lecture 17, normalizing flows in UDL chapter 16)
and by stating the problem they all try to solve.


## Four families of generative model

![*[L. Weng, "What are Diffusion Models?", blog post, 2021](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/)*](figs/18-Diffusion-and-Flow-Matching/generative-model-families.png)

**Given a distribution known only through samples, how do we generate a new sample?**
Every family learns a map from a simple distribution we can sample (Gaussian
noise) to the complex one we cannot. Diffusion models learn that map as many
small steps; flow matching, later today, learns it as an ODE.

## A short history

- 2013: variational autoencoders ([Kingma & Welling](https://arxiv.org/abs/1312.6114)); 2014: GANs ([Goodfellow et al.](https://arxiv.org/abs/1406.2661))
- 2015: diffusion as non-equilibrium thermodynamics ([Sohl-Dickstein et al., ICML 2015](https://arxiv.org/abs/1503.03585)); the U-Net ([Ronneberger et al., MICCAI 2015](https://arxiv.org/abs/1505.04597))
- 2019: score-based generative modeling ([Song & Ermon, NeurIPS 2019](https://arxiv.org/abs/1907.05600))
- 2020: DDPM, the noise-prediction objective ([Ho, Jain & Abbeel, NeurIPS 2020](https://arxiv.org/abs/2006.11239))
- 2021: DDIM ([Song, Meng & Ermon, ICLR 2021](https://arxiv.org/abs/2010.02502)); the SDE unification ([Song et al., ICLR 2021](https://arxiv.org/abs/2011.13456)); classifier guidance ([Dhariwal & Nichol](https://arxiv.org/abs/2105.05233)); latent diffusion posted to arXiv in December ([Rombach et al.](https://arxiv.org/abs/2112.10752))
- 2022: latent diffusion at CVPR, released as Stable Diffusion; classifier-free guidance ([Ho & Salimans](https://arxiv.org/abs/2207.12598))
- 2023: Diffusion Transformer ([Peebles & Xie, ICCV 2023](https://arxiv.org/abs/2212.09748)); flow matching ([Lipman et al., ICLR 2023](https://arxiv.org/abs/2210.02747)) and rectified flow ([Liu, Gong & Liu, ICLR 2023](https://arxiv.org/abs/2209.03003)); consistency models ([Song et al., ICML 2023](https://arxiv.org/abs/2303.01469))
- 2024: Stable Diffusion 3 and Flux put rectified flow in production ([Esser et al., ICML 2024](https://arxiv.org/abs/2403.03206))
- 2025: discrete diffusion for text at LLM scale ([LLaDA, Nie et al.](https://arxiv.org/abs/2502.09992); Gemini Diffusion)


## The basic idea of a diffusion probabilistic model

- Define a **fixed stochastic forward process** that progressively destroys information, taking data from the complex target distribution to a simple Gaussian.
- **Learn the reverse process**, which takes Gaussian noise as input and produces data from the distribution of interest.
- Learn it as one network $G(\text{image}, \text{step})$ shared across steps, not $T$ separate networks and not one giant jump $G^*$.


![Rocca, 2022](figs/18-Diffusion-and-Flow-Matching/one-step-vs-multi-step.png)


# Diffusion processes

Before the model itself we need some vocabulary for random processes: the
Markov property, the Wiener process, and diffusion processes described by
stochastic differential equations. The payoff is one theorem: a diffusion run
backwards in time is again a diffusion, whose drift involves the gradient of
the log density. That gradient, the score, is what every model in this lecture
learns in one form or another.


## Stochastic and Markov processes

A **stochastic process** is a random variable indexed by time: discrete $X_n$, $n \in \mathbb{N}$, or continuous $X_t$, $t \ge 0$. A realization of a random variable is a *sample*; a realization of a process is a *sample path* or *trajectory*.

A **Markov process** has no memory of the past beyond the present:

$$
P\left(X_{t_n} \middle| X_{t_{n-1}}, \ldots, X_{t_0}\right) = P\left(X_{t_n} \middle| X_{t_{n-1}}\right) \quad \forall\, t_0 < t_1 < \ldots < t_n
$$

i.i.d. coin flips are trivially Markov; a random walk is Markov; stock prices with momentum and natural language are not.



## Diffusion processes and SDEs

Any *diffusion process* can be described by a *stochastic differential equation* (SDE)


$$
\underbrace{dX_t = a(X_t, t)\,dt}_{\text{ordinary differential equation}} + \underbrace{\sigma(X_t, t)\,dW_t}_{\text{stochastic part}}
$$

An infinitesimal change in $X$ equals $a$ times an infinitesimal change in time plus $\sigma$ times an infinitesimal increment of a Wiener process.


- $a(\cdot)$ is the **drift** coefficient, $\sigma(\cdot)$ the **diffusion** coefficient; both may depend on the value and on time
- $W_t$ is the **Wiener process**


## The Wiener process (Brownian motion)

A continuous-time stochastic process $W_t$ with:

1. $W_0 = 0$ almost surely
2. **Independent increments:** for every $t > 0$, the future increments $W_{t+u} - W_t$, $u \geq 0$, are independent of the past values $W_s$, $s < t$
3. **Gaussian increments:** $W_{t+u} - W_t$ is normally distributed with mean 0 and variance $u$, $\boxed{W_{t+u} - W_t \sim \mathcal{N}(0, u)}$
4. **Continuous paths** almost surely

![Five sample paths; the shaded band is one standard deviation, $\sqrt{t}$. <https://en.wikipedia.org/wiki/Wiener_process>](figs/18-Diffusion-and-Flow-Matching/wiener-process-samples.png)

## Norbert Wiener

**Norbert Wiener** (November 26, 1894 – March 18, 1964) was an American mathematician and philosopher, professor of mathematics at MIT.

A child prodigy, Wiener became *an early researcher in stochastic and mathematical noise processes*, contributing to electronic engineering, communication and control systems.

He is considered the originator of cybernetics, the science of communication as it relates to living things and machines, and heavily influenced John von Neumann, Claude Shannon and others.

Wrote "[The Machine Age](https://cdn.libraries.mit.edu/dissemination/diponline/MC0022/MC0022_MachineAgeV3_1949.pdf)" in 1949, anticipating robots and automation.


![Norbert Wiener is the instructor's "great, great grand advisor".](figs/18-Diffusion-and-Flow-Matching/norbert-wiener.png)


[Wikipedia](https://en.wikipedia.org/wiki/Norbert_Wiener); [New York Times, 2013, on the rediscovered 1949 essay](https://www.nytimes.com/2013/05/21/science/mit-scholars-1949-essay-on-machine-age-is-found.html)

## Discretizing the SDE

Over a short interval $dt$ the Wiener increment is

$$
dW_t \approx W_{t+dt} - W_t \sim \mathcal{N}(0, dt),
$$

so its **standard deviation is $\sqrt{dt}$**: over a short step the noise, of size $\sqrt{dt}$, dominates the drift, of size $dt$. Discretizing the SDE,

$$
X_{t+dt} - X_t \approx a(X_t, t)\,dt + \sigma(X_t, t)\,U \quad \text{where} \quad U \sim \mathcal{N}(0, dt)
$$

which can be rewritten as

$$
X_{t+dt} \approx X_t + \underbrace{a(X_t, t)\,dt}_{\text{deterministic drift}} + U' \quad \text{where} \quad U' \sim \mathcal{N}\!\left(0,\ \sigma^2(X_t, t)\,dt\right)
$$

Scaling a Gaussian by $\sigma$ scales its variance by $\sigma^2$, so the noise term has variance $\sigma^2 dt$ and standard deviation $\sigma\sqrt{dt}$. This update rule is the Euler–Maruyama scheme; the DDPM forward step is one instance of it.


Adapted from Rocca, 2022.


## Drift gives the trend, diffusion the randomness

![Sample paths for several SDEs and the matching (non-stochastic) ODE. Rocca, 2022.](figs/18-Diffusion-and-Flow-Matching/diffusion-process-samples.png)


## Reversing time: Anderson's theorem

Let $X_t$ solve $dX_t = a(X_t, t)\,dt + \sigma(t)\,dW_t$ on $[0, T]$ and let $p_t$ be the marginal density of $X_t$. [Anderson (1982)](https://doi.org/10.1016/0304-4149(82)90051-5) showed that the time reversal is also a diffusion. In the form used by [Song et al. (2021)](https://arxiv.org/abs/2011.13456), eq. 6,

$$
dX_t = \left[a(X_t, t) - \sigma^2(t)\,\nabla_x \log p_t(X_t)\right]dt + \sigma(t)\,d\bar{W}_t,
$$

where $\bar{W}_t$ is a Wiener process that runs backwards from $T$ to $0$ and $dt$ is a *negative* infinitesimal step.

Equivalently, with a clock that runs forward on the reversed process, $\bar{X}_s = X_{T-s}$,

$$
d\bar{X}_s = \left[-a(\bar{X}_s, T-s) + \sigma^2(T-s)\,\nabla_x \log p_{T-s}(\bar{X}_s)\right]ds + \sigma(T-s)\,dW_s .
$$

The drift flips sign and gains a term that pushes **uphill in log density**.


$\nabla_x \log p_t(x)$ is the **score function**. We do not know $p_t$, so we will estimate the score with a neural network. The score is easier to learn than $p_t$ itself: it does not need to integrate to one, so the normalizing constant never appears.



# The forward process and the diffusion kernel

The SDE vocabulary tells us what to expect. Now we build the forward process
as a discrete Markov chain with Gaussian transitions, and derive its most
useful property: we can jump from $x_0$ to any $x_t$ in one step.


## Shrink and add noise

A shrinking drift with a non-zero diffusion coefficient turns any distribution into an isotropic Gaussian:

$$
X_t = \sqrt{1-p}\,X_{t-1} + \sqrt{p}\,u_t \qquad u_t \sim \mathcal{N}(0, 1), \quad p = 0.01
$$

![Rocca, 2022](figs/18-Diffusion-and-Flow-Matching/diffusion-towards-gaussian.png) + sqrt(0.01) u_t, where u_t ~ N(0,1), X_t -> N(0,1). On the left a bimodal starting density with modes near 2 and -2; many sample paths run from step 0 to 200, mixing together; on the right the final density is a standard normal centered at 0." width="60%"}


## Why any start ends at $\mathcal{N}(0, 1)$

Unroll the recurrence $T$ times:

$$
\begin{aligned}
X_T &= \sqrt{1-p}\,X_{T-1} + \sqrt{p}\,u_T \\
&= (\sqrt{1-p})^2 X_{T-2} + \sqrt{1-p}\sqrt{p}\,u_{T-1} + \sqrt{p}\,u_T \\
&= (\sqrt{1-p})^T X_0 + \underbrace{\sum_{i=0}^{T-1} \sqrt{p}\,(\sqrt{1-p})^i\, u_{T-i}}_{\text{sum of independent Gaussians}}
\end{aligned}
$$

A sum of independent Gaussians is Gaussian with the variances added. For large $T$,

$$
(\sqrt{1-p})^T \underset{T\to\infty}{\longrightarrow} 0 \qquad \text{and} \qquad \underbrace{\sum_{i=0}^{T-1} \overbrace{\left(\sqrt{p}(\sqrt{1-p})^i\right)^2}^{\text{geometric series}}}_{\text{variance}} = p\,\frac{1-(1-p)^T}{1-(1-p)} \underset{T\to\infty}{\longrightarrow} 1
$$

**Whatever $X_0$ was, $X_T$ tends to a standard Gaussian.** In $H \times W \times C$ dimensions the same holds pixel by pixel.


Adapted from Rocca, 2022.


## The forward process as a Markov chain

Notation: $\mathcal{N}(x; \mu, \Sigma)$. Let $x_0 \sim q(x_0)$ be a data sample. The forward process is a Markov chain with Gaussian transitions,

$$
q(x_t \mid x_{t-1}) = \mathcal{N}\left(x_t;\ \sqrt{1-\beta_t}\, x_{t-1},\ \beta_t I\right), \qquad t = 1, \ldots, T,
$$

where the **noise schedule** $\beta_t \in (0,1)$ sets the trade-off between information kept from the previous step and noise added. Equivalently, with $\alpha_t = 1 - \beta_t$,

$$
x_t = \sqrt{\alpha_t}\, x_{t-1} + \sqrt{1 - \alpha_t}\, \epsilon_t, \qquad \epsilon_t \sim \mathcal{N}(0, I),
$$

the discretized shrink-and-add-noise process from before, with $p = \beta_t$ (a reparameterized sample, as in lecture 17). By the Markov property the whole trajectory has density $q(x_{1:T} \mid x_0) = \prod_{t=1}^{T} q(x_t \mid x_{t-1})$. [Ho et al. (2020)](https://arxiv.org/abs/2006.11239) used $T = 1000$ with $\beta_t$ increasing linearly from $10^{-4}$ to $0.02$.


## The diffusion kernel $q(x_t \mid x_0)$

*Quizzable.* Substitute the step for $x_{t-1}$ into the step for $x_t$:

$$
\begin{aligned}
x_t &= \sqrt{\alpha_t}\left(\sqrt{\alpha_{t-1}}\,x_{t-2} + \sqrt{1-\alpha_{t-1}}\,\epsilon_{t-1}\right) + \sqrt{1-\alpha_t}\,\epsilon_t \\
&= \sqrt{\alpha_t\alpha_{t-1}}\,x_{t-2} + \underbrace{\sqrt{\alpha_t(1-\alpha_{t-1})}\,\epsilon_{t-1} + \sqrt{1-\alpha_t}\,\epsilon_t}_{\text{independent Gaussians: variance } \alpha_t(1-\alpha_{t-1}) + (1-\alpha_t) = 1 - \alpha_t\alpha_{t-1}}
\end{aligned}
$$

Two Gaussian steps compose into one Gaussian step with the signal coefficient multiplied and the noise variance one minus the signal variance. By induction down to $x_0$, with $\bar\alpha_t = \prod_{i=1}^{t}\alpha_i$,

$$
\boxed{\;q(x_t \mid x_0) = \mathcal{N}\left(x_t;\ \sqrt{\bar\alpha_t}\,x_0,\ (1-\bar\alpha_t)\,I\right)
\quad\Longleftrightarrow\quad
x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,\ \ \epsilon \sim \mathcal{N}(0, I)\;}
$$

We can sample any $x_t$ directly from $x_0$ in one line, which is what makes training cheap. As $t \to T$, $\bar\alpha_t \to 0$ and $x_T$ is pure noise. $\sqrt{\bar\alpha_t}$ is the signal level and $\sqrt{1-\bar\alpha_t}$ the noise level; their ratio sets the signal-to-noise ratio of step $t$.


Run the forward chain on the 1D toy data with the schedule of Ho et al., one
step at a time, and check the closed form against it. The data is a mixture of
$\mathcal{N}(-2, 0.35^2)$ and $\mathcal{N}(1.5, 0.5^2)$ with weights 0.35 and
0.65. Each mixture component is Gaussian, so the noised marginal $q_t(x_t)$ is
again a two-component mixture, with means $\sqrt{\bar\alpha_t}\,\mu_k$ and
variances $\bar\alpha_t\sigma_k^2 + 1 - \bar\alpha_t$.

In [ ]:
#| code-fold: false
# Adapted from UDL notebook 18.1 (MIT)
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#dcdcd8", "grid.linewidth": 0.6, "legend.frameon": False})

MIX_W = np.array([0.35, 0.65])     # the toy data distribution q(x_0)
MIX_MU = np.array([-2.0, 1.5])
MIX_SD = np.array([0.35, 0.5])

def sample_data(n, rng):
    k = rng.choice(2, size=n, p=MIX_W)
    return MIX_MU[k] + MIX_SD[k] * rng.normal(size=n)

T = 1000
betas = np.linspace(1e-4, 0.02, T)          # beta_1 ... beta_T (index t-1)
alphas = 1 - betas
alpha_bars = np.cumprod(alphas)
print(f"alpha_bar_T = {alpha_bars[-1]:.2e}, signal level sqrt(alpha_bar_T) = "
      f"{np.sqrt(alpha_bars[-1]):.4f}")

rng = np.random.default_rng(0)
n = 50_000
x = sample_data(n, rng)                     # a batch of data samples
x0_fixed = 1.5
y = np.full(n, x0_fixed)                    # n copies of one x_0, for q(x_t | x_0)
snapshots = {0: x.copy()}
print(f"\n{'t':>5} {'chain mean':>11} {'closed form':>12} {'chain var':>10} {'closed form':>12}")
for t in range(1, T + 1):
    x = np.sqrt(alphas[t - 1]) * x + np.sqrt(betas[t - 1]) * rng.normal(size=n)
    y = np.sqrt(alphas[t - 1]) * y + np.sqrt(betas[t - 1]) * rng.normal(size=n)
    if t in (10, 50, 100, 300, 1000):
        snapshots[t] = x.copy()
        ab = alpha_bars[t - 1]
        print(f"{t:>5} {y.mean():>11.4f} {np.sqrt(ab) * x0_fixed:>12.4f} "
              f"{y.var():>10.4f} {1 - ab:>12.4f}")

Running the one-step chain $t$ times from $x_0 = 1.5$ gives the mean
$\sqrt{\bar\alpha_t}\,x_0$ and variance $1 - \bar\alpha_t$ of the diffusion
kernel, to Monte Carlo error. Below, the whole data distribution: histograms
of the chain against the exact marginal $q_t$.

In [ ]:
#| code-fold: true
#| fig-cap: "The forward process on the 1D toy data. Histograms: 50,000 samples run through the chain. Solid: the exact marginal $q_t$. Dashed: $\\mathcal{N}(0, 1)$."
#| fig-alt: "Five histograms side by side for t = 0, 50, 100, 300 and 1000. At t = 0 two separated peaks near -2 and 1.5; at t = 50 and 100 the peaks move toward zero and widen; at t = 300 they have merged into a skewed bump with a shoulder on the left; at t = 1000 the histogram matches the dashed standard normal curve. A solid curve follows each histogram."
def gauss(x, mean, var):
    return np.exp(-0.5 * (x - mean) ** 2 / var) / np.sqrt(2 * np.pi * var)

def marginal_params(ab):
    """Means and variances of the mixture components of q_t, given alpha_bar_t."""
    return np.sqrt(ab) * MIX_MU, ab * MIX_SD ** 2 + 1 - ab

def marginal_pdf(x, ab):
    means, variances = marginal_params(ab)
    return sum(w * gauss(x, m, v) for w, m, v in zip(MIX_W, means, variances))

grid = np.linspace(-4, 4, 400)
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6), sharey=True)
for ax, t in zip(axes, [0, 50, 100, 300, 1000]):
    ab = 1.0 if t == 0 else alpha_bars[t - 1]
    ax.hist(snapshots[t], bins=80, range=(-4, 4), density=True, color="#9ec5f0")
    ax.plot(grid, marginal_pdf(grid, ab), color="#2a78d6", lw=1.6)
    ax.plot(grid, gauss(grid, 0, 1), color="#1d1d1b", lw=1, ls="--")
    ax.set_title(f"$t = {t}$, $\\bar\\alpha_t = {ab:.3f}$", fontsize=10)
    ax.set_xlabel("$x_t$")
axes[0].set_ylabel("density")
fig.tight_layout()
plt.show()

# The reverse process and the objective

The forward chain is fixed. Generation needs the reverse chain, whose
transitions we approximate with Gaussians produced by a network, and a loss
that makes the reverse chain produce samples from $q(x_0)$. The loss comes
from the same ELBO argument as the VAE in lecture 17, with $T$ latents instead
of one.


## The reverse process

For small $\beta_t$ the true reverse transition $q(x_{t-1} \mid x_t)$ is approximately Gaussian (a classical result, Feller 1949, used by [Sohl-Dickstein et al. 2015](https://arxiv.org/abs/1503.03585); Anderson 1982 is the continuous-time version), so we approximate it with a Gaussian whose parameters come from a network:

$$
p_\theta(x_{t-1} \mid x_t) = \mathcal{N}\left(x_{t-1};\ \mu_\theta(x_t, t),\ \Sigma_\theta(x_t, t)\right).
$$

By the Markov property a backward trajectory has density

$$
p_\theta(x_{0:T}) = p(x_T) \prod_{t=1}^{T} p_\theta(x_{t-1} \mid x_t), \qquad p(x_T) = \mathcal{N}(x_T; 0, I),
$$

where the starting distribution $p(x_T)$ is an isotropic Gaussian that does not depend on $\theta$, chosen to match where the forward process ends.


## Fixed forward chain, learned backward chain

![Rocca, 2022. The figure's final box should read $p(x_T) = \mathcal{N}(x_T; 0, I)$.](figs/18-Diffusion-and-Flow-Matching/forward-backward-process.png)) = N(x_t; sqrt(1-beta_t) x_{t-1}, beta_t I). Cherry blossom images are noised step by step with arrows q(x_1|x_0), q(x_2|x_1), ..., q(x_T|x_{T-1}). Learned backward process: arrows p_theta(x_{T-1}|x_T), ..., p_theta(x_0|x_1); approximation of q(x_{t-1}|x_t); gaussian transition kernel with parameters to be learned p_theta(x_{t-1}|x_t) = N(x_{t-1}; mu_theta(x_t,t), Sigma_theta(x_t,t)); initial distribution p(x_T) = N(x_T; 0, I)." width="80%"}

## What to optimize: from KL to the ELBO

We want the end of the reverse chain, $p_\theta(x_0)$, to match the data, $q(x_0)$:

$$
\theta^* = \arg\min_\theta D_{KL}\!\left(q(x_0)\,\|\,p_\theta(x_0)\right)
= \arg\min_\theta \mathbb{E}_{q(x_0)}\left[-\log p_\theta(x_0)\right] .
$$

$p_\theta(x_0) = \int p_\theta(x_{0:T})\,dx_{1:T}$ is intractable, so, exactly as for the VAE, use the forward chain as the variational posterior and apply Jensen's inequality to get an upper bound on the negative log-likelihood ([Ho et al. 2020](https://arxiv.org/abs/2006.11239), eq. 3):

$$
\mathbb{E}_q\left[-\log p_\theta(x_0)\right] \le
\mathbb{E}_q\left[-\log p(x_T) - \sum_{t\ge1}\log\frac{p_\theta(x_{t-1} \mid x_t)}{q(x_t \mid x_{t-1})}\right] =: L .
$$


Rewriting with Bayes' rule, $L$ becomes a sum of KL divergences between Gaussians (eq. 5):

$$
L = \mathbb{E}_q\Big[\underbrace{D_{KL}\big(q(x_T \mid x_0)\,\|\,p(x_T)\big)}_{L_T:\ \text{no parameters}} + \sum_{t>1}\underbrace{D_{KL}\big(q(x_{t-1} \mid x_t, x_0)\,\|\,p_\theta(x_{t-1} \mid x_t)\big)}_{L_{t-1}} \underbrace{- \log p_\theta(x_0 \mid x_1)}_{L_0}\Big]
$$


## The forward posterior is Gaussian and known

Each $L_{t-1}$ compares the network's Gaussian with $q(x_{t-1} \mid x_t, x_0)$: the forward chain run backwards *given the clean image*. Since every factor is Gaussian, this posterior is Gaussian too (Ho et al., eq. 6–7):

$$
q(x_{t-1} \mid x_t, x_0) = \mathcal{N}\left(x_{t-1};\ \tilde\mu_t(x_t, x_0),\ \tilde\beta_t I\right),
$$

$$
\tilde\mu_t(x_t, x_0) = \frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}\,x_0 + \frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\,x_t,
\qquad
\tilde\beta_t = \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\,\beta_t .
$$

Fix the reverse variance to a schedule, $\Sigma_\theta = \rho_t^2 I$ (Ho et al. write $\sigma_t^2$ and use $\beta_t$ or $\tilde\beta_t$; we keep $\sigma_t$ for the noise level $\sqrt{1-\bar\alpha_t}$ of section 6). The KL between two Gaussians with equal covariance is a scaled squared distance between means (eq. 8):

$$
L_{t-1} = \mathbb{E}_q\left[\frac{1}{2\rho_t^2}\left\|\tilde\mu_t(x_t, x_0) - \mu_\theta(x_t, t)\right\|^2\right] + C .
$$

So the network's job is to predict the posterior mean. The last step is to choose *what* it outputs.


## From the mean to noise prediction

Write $x_0$ in terms of the noise that produced $x_t$: $x_0 = \left(x_t - \sqrt{1-\bar\alpha_t}\,\epsilon\right)/\sqrt{\bar\alpha_t}$. Substituting into $\tilde\mu_t$ and simplifying (eq. 10),

$$
\tilde\mu_t = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\epsilon\right).
$$

Everything on the right is known at training time except $\epsilon$. So let the network predict the noise, $\epsilon_\theta(x_t, t)$, and set $\mu_\theta$ to the same expression with $\epsilon_\theta$ in place of $\epsilon$. Then (eq. 12)

$$
L_{t-1} - C = \mathbb{E}_{x_0, \epsilon}\left[\frac{\beta_t^2}{2\rho_t^2\,\alpha_t\,(1-\bar\alpha_t)}\left\|\epsilon - \epsilon_\theta\!\left(\sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,\ t\right)\right\|^2\right].
$$


[Ho et al.](https://arxiv.org/abs/2006.11239) found that **dropping the weight** works better (eq. 14):

$$
\boxed{\;L_{\text{simple}}(\theta) = \mathbb{E}_{t,\,x_0,\,\epsilon}\left[\left\|\epsilon - \epsilon_\theta\!\left(\sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon,\ t\right)\right\|^2\right]\;}
$$

$L_{\text{simple}}$ is a *simplified objective*: a reweighted ELBO that down-weights the small-$t$ terms, not itself a bound on the likelihood. Training a diffusion model is a regression from a noisy image to the noise in it.


# Training, sampling and the U-Net

The simplified objective turns training into noise prediction: a network
$\epsilon_\theta(x_t, t)$ learns to estimate the noise that was added to an
image at step $t$, and sampling runs that network backwards from pure noise.


## Algorithm 1: training

1. **repeat**
2. $\quad x_0 \sim q(x_0)$: *a random training image*
3. $\quad t \sim \mathrm{Uniform}(\{1, \ldots, T\})$: *a random step*
4. $\quad \epsilon \sim \mathcal{N}(0, I)$: *random noise*
5. $\quad x_t = \sqrt{\bar{\alpha}_t}\,x_0 + \sqrt{1-\bar{\alpha}_t}\,\epsilon$: *jump straight to step $t$ with the diffusion kernel*
6. $\quad$ gradient step on $\nabla_\theta \left\|\epsilon - \epsilon_\theta(x_t, t)\right\|^2$
7. **until** converged

![Rocca, 2022](figs/18-Diffusion-and-Flow-Matching/training-process.png)

## Algorithm 2: sampling (unconditional)

1. $x_T \sim \mathcal{N}(0, I)$: *start from isotropic Gaussian noise*
2. **for** $t = T, \ldots, 1$ **do**
3. $\quad z \sim \mathcal{N}(0, I)$ if $t > 1$ else $z = 0$: *fresh noise, except at the last step*
4. $\quad \tilde{\epsilon} = \epsilon_\theta(x_t, t)$: *estimated noise in the current image*
5. $\quad \tilde{x}_0 = \frac{1}{\sqrt{\bar{\alpha}_t}}\left(x_t - \sqrt{1-\bar{\alpha}_t}\,\tilde{\epsilon}\right)$: *implied estimate of the clean image*
6. $\quad \tilde{\mu} = \tilde\mu_t(x_t, \tilde{x}_0) = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\tilde\epsilon\right)$: *posterior mean of the previous step*
7. $\quad x_{t-1} = \tilde{\mu} + \rho_t z$: *sample the previous step*
8. **end for**
9. **return** $x_0$

> $\epsilon_\theta(x_t, t)$ is the trained network; $\rho_t$ is the **standard deviation** of the reverse step, from the schedule ($\rho_t^2 = \beta_t$ or $\tilde\beta_t$).
>
> Each iteration moves a little toward the current guess of $\tilde x_0$ and re-adds a little noise. $T = 1000$ network evaluations per image: slow, which sections 6 and 7 fix.

Adapted from Rocca, 2022.

Algorithm 2 needs a trained $\epsilon_\theta$. On the toy data we can skip
training and use the best possible network: least squares converges to the
conditional mean, so a perfectly trained $\epsilon_\theta(x_t, t)$ outputs
$\mathbb{E}[\epsilon \mid x_t]$. Given the mixture component $k$, $x_t$ and
$\epsilon$ are jointly Gaussian with covariance $\sigma_t = \sqrt{1-\bar\alpha_t}$
(the noise level, not the reverse-step $\rho_t$),
so $\mathbb{E}[\epsilon \mid x_t, k] = \sigma_t (x_t - m_k)/v_k$ with $m_k$, $v_k$
the component's mean and variance under $q_t$; averaging over the posterior
probability $r_k(x_t)$ of each component gives the exact predictor. The
sampling loop is lines 1–9, with $\rho_t^2 = \beta_t$.

In [ ]:
#| code-fold: false
# Adapted from UDL notebooks 18.2 and 18.3 (MIT)
def responsibilities(x, ab):
    """Posterior probability r_k(x_t) of each mixture component under q_t."""
    means, variances = marginal_params(ab)
    logp = (np.log(MIX_W)[:, None] - 0.5 * np.log(2 * np.pi * variances)[:, None]
            - 0.5 * (x[None, :] - means[:, None]) ** 2 / variances[:, None])
    p = np.exp(logp - logp.max(axis=0))
    return p / p.sum(axis=0), means, variances

def eps_ideal(x, t):
    """E[eps | x_t]: the output of a perfectly trained noise predictor."""
    ab = alpha_bars[t - 1]
    r, means, variances = responsibilities(x, ab)
    return np.sqrt(1 - ab) * (r * (x[None, :] - means[:, None]) / variances[:, None]).sum(axis=0)

rng = np.random.default_rng(1)
x = rng.normal(size=20_000)                                     # line 1: x_T ~ N(0, 1)
for t in range(T, 0, -1):                                       # line 2
    z = rng.normal(size=x.shape) if t > 1 else 0.0              # line 3
    eps = eps_ideal(x, t)                                       # line 4
    mu = (x - betas[t - 1] / np.sqrt(1 - alpha_bars[t - 1]) * eps) / np.sqrt(alphas[t - 1])  # line 6
    x = mu + np.sqrt(betas[t - 1]) * z                          # line 7
samples = x

left = samples < 0
print(f"fraction in the left mode: {left.mean():.3f}  (data: {MIX_W[0]})")
print(f"left mode:  mean {samples[left].mean():+.3f}, sd {samples[left].std():.3f}  "
      f"(data: {MIX_MU[0]:+.1f}, {MIX_SD[0]})")
print(f"right mode: mean {samples[~left].mean():+.3f}, sd {samples[~left].std():.3f}  "
      f"(data: {MIX_MU[1]:+.1f}, {MIX_SD[1]})")

In [ ]:
#| code-fold: true
#| fig-cap: "1,000 steps of Algorithm 2 with the ideal noise predictor turn 20,000 draws of $\\mathcal{N}(0, 1)$ into samples of the data distribution."
#| fig-alt: "Histogram of the generated samples with two peaks, a narrow one near -2 and a wider, taller one near 1.5, closely following a solid curve showing the true bimodal data density; a dashed standard normal curve shows where the samples started."
fig, ax = plt.subplots(figsize=(7, 2.8))
ax.hist(samples, bins=100, range=(-4, 4), density=True, color="#9ec5f0", label="Algorithm 2 samples")
ax.plot(grid, marginal_pdf(grid, 1.0), color="#2a78d6", lw=1.6, label="data $q(x_0)$")
ax.plot(grid, gauss(grid, 0, 1), color="#1d1d1b", lw=1, ls="--", label="start $\\mathcal{N}(0, 1)$")
ax.set_xlabel("$x_0$")
ax.set_ylabel("density")
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

Swap `eps_ideal` for a network and this is the whole DDPM sampler. Set
`z = 0.0` at every step and all 20,000 samples collapse to nearly one point:
the fresh noise is what keeps the variance right. The deterministic sampler
that works is a different update, the probability-flow ODE of section 6.


## The U-Net (2015) as the noise estimator

![](figs/18-Diffusion-and-Flow-Matching/unet-noise-estimation.png)

The output has the same shape as the input. The encoder-decoder with skip connections is the U-Net of [Ronneberger et al. (2015)](https://arxiv.org/abs/1505.04597), with residual blocks and self-attention at low resolution added by Ho et al.


![](figs/18-Diffusion-and-Flow-Matching/time-embedding-feature-maps.png)

**Time embedding:** $t$ goes through a sinusoidal encoding (as in Transformers) and a two-layer MLP; each residual block adds its own linear projection of the embedding to its feature map, so the one network knows which step it is denoising.


# Score matching and flow matching

DDPM predicts noise. Two other research lines, score matching and continuous
normalizing flows, arrived at the same model from different directions, and by
2023 they had merged into one framework. This section goes through it in three
steps: the noise the network predicts *is* the score, up to a sign and a
scale; the discrete chain is a discretization of an SDE with an equivalent
deterministic ODE; and that ODE can be learned directly, on straight paths,
which is flow matching. Each step is one slide of derivation.


## The score function

The **score** of a density is the gradient of its log with respect to the *data*,

$$
s(x) = \nabla_x \log p(x).
$$

- If $p(x) = \tilde p(x) / Z$, then $\nabla_x \log p = \nabla_x \log \tilde p$: the normalizing constant $Z$, the thing that makes likelihood-based training of energy models intractable, disappears.
- Knowing the score is enough to sample. **Langevin dynamics** $x_{k+1} = x_k + \frac{\eta}{2}\,s(x_k) + \sqrt{\eta}\,z_k$, $z_k \sim \mathcal{N}(0, I)$, is an SDE discretization whose stationary distribution is $p$: climb the log density, jitter, repeat.
- [Song & Ermon (2019)](https://arxiv.org/abs/1907.05600) trained a network $s_\theta(x, \sigma)$ to estimate the score of the data *blurred with Gaussian noise* at several noise levels $\sigma$, and sampled with annealed Langevin dynamics from large $\sigma$ to small. Large noise fills in the score where there is no data; small noise gives the detail.
- That is a diffusion model described in a different language. The next slide shows the dictionary.


## Noise and score are the same thing

*Quizzable.* The diffusion kernel is Gaussian, $q(x_t \mid x_0) = \mathcal{N}\!\left(\sqrt{\bar\alpha_t}\,x_0,\ (1-\bar\alpha_t) I\right)$, so its score has a closed form:

$$
\nabla_{x_t} \log q(x_t \mid x_0)
= -\frac{x_t - \sqrt{\bar\alpha_t}\,x_0}{1-\bar\alpha_t}
= -\frac{\sqrt{1-\bar\alpha_t}\,\epsilon}{1-\bar\alpha_t}
= -\frac{\epsilon}{\sigma_t}, \qquad \sigma_t := \sqrt{1-\bar\alpha_t}.
$$

So $\boxed{\epsilon = -\sigma_t\, s}$: the noise is minus the score scaled by the noise level, and

$$
\left\|\epsilon - \epsilon_\theta(x_t, t)\right\|^2 = \sigma_t^2\left\|\nabla_{x_t}\log q(x_t \mid x_0) - s_\theta(x_t, t)\right\|^2
\quad\text{with}\quad s_\theta := -\epsilon_\theta / \sigma_t .
$$

$L_{\text{simple}}$ is **denoising score matching** ([Vincent 2011](https://doi.org/10.1162/NECO_a_00142)) with weight $\sigma_t^2$. A trained DDPM is a score model, and a trained score model is a DDPM.


But the loss regresses on the score of the *conditional* $q(x_t \mid x_0)$, while sampling needs the score of the *marginal* $q_t(x_t)$. Why is that fine?


## Why the conditional target gives the marginal score

Write the marginal as a mixture over clean images, $q_t(x) = \int q(x \mid x_0)\,q(x_0)\,dx_0$. Then

$$
\nabla_x \log q_t(x) = \frac{\nabla_x q_t(x)}{q_t(x)}
= \int \underbrace{\frac{q(x \mid x_0)\,q(x_0)}{q_t(x)}}_{q(x_0 \mid x)} \nabla_x \log q(x \mid x_0)\,dx_0
= \mathbb{E}_{q(x_0 \mid x)}\!\left[\nabla_x \log q(x \mid x_0)\right].
$$

- The marginal score is the **posterior average** of the conditional scores. Least-squares regression onto a random target converges to the target's conditional mean, so the minimizer of the conditional loss is exactly $\nabla_x \log q_t(x)$. Same for noise: $\epsilon_\theta^*(x_t, t) = \mathbb{E}[\epsilon \mid x_t]$.
- Consequence (**Tweedie's formula**): $\mathbb{E}[x_0 \mid x_t] = \left(x_t + \sigma_t^2\, s(x_t)\right)/\sqrt{\bar\alpha_t}$, which is line 5 of Algorithm 2. The network's $\tilde x_0$ is the posterior mean of the clean image, the blurry "best guess" that gets sharpened over the steps.
- Remember the shape of this argument: *regress onto a cheap per-sample target, get the expensive marginal quantity for free.* Flow matching uses it again in four slides.


Check all three claims on the toy data at $t = 300$. (1) Per sample,
$\epsilon = -\sigma_t \nabla_{x_t}\log q(x_t \mid x_0)$ exactly. (2) The marginal
score $s(x) = \nabla_x \log q_t(x)$, taken here by finite differences of the
exact $\log q_t$, satisfies $\epsilon^*(x) = -\sigma_t\,s(x)$ for the ideal
predictor `eps_ideal` used in Algorithm 2. (3) Averaging the per-sample targets
$\epsilon$ over samples with nearly the same $x_t$ (what least squares does)
recovers $-\sigma_t\,s(x_t)$.

In [ ]:
#| code-fold: false
t = 300
ab = alpha_bars[t - 1]
sigma_t = np.sqrt(1 - ab)
rng = np.random.default_rng(2)
x0 = sample_data(1_000_000, rng)
eps = rng.normal(size=x0.size)
xt = np.sqrt(ab) * x0 + sigma_t * eps

# (1) the conditional score of q(x_t | x_0) and the noise
cond_score = -(xt - np.sqrt(ab) * x0) / (1 - ab)
print(f"(1) max |eps + sigma_t * cond_score|     = {np.max(np.abs(eps + sigma_t * cond_score)):.1e}")

# (2) the marginal score by central differences of log q_t, against eps_ideal
def score(x, ab, h=1e-5):
    return (np.log(marginal_pdf(x + h, ab)) - np.log(marginal_pdf(x - h, ab))) / (2 * h)

xs = np.linspace(-3.5, 3.5, 701)
print(f"(2) max |eps_ideal + sigma_t * score|    = "
      f"{np.max(np.abs(eps_ideal(xs, t) + sigma_t * score(xs, ab))):.1e}")

# (3) least squares = conditional mean: average eps in narrow bins of x_t
edges = np.linspace(-3, 3, 61)
centers = 0.5 * (edges[:-1] + edges[1:])
idx = np.digitize(xt, edges) - 1
inside = (idx >= 0) & (idx < 60)
counts = np.bincount(idx[inside], minlength=60)
bin_mean = np.bincount(idx[inside], weights=eps[inside], minlength=60) / np.maximum(counts, 1)
well_filled = counts >= 2000
gap = np.abs(bin_mean + sigma_t * score(centers, ab))[well_filled]
print(f"(3) max |binned mean of eps + sigma_t * score| over {well_filled.sum()} bins = {gap.max():.3f}")

The first two agree to rounding and finite-difference error. The third is a
Monte Carlo regression with a few thousand to a few tens of thousands of
samples per bin, so it agrees only to a couple of hundredths; more samples
shrink the gap like $1/\sqrt{n}$.

In [ ]:
#| code-fold: true
#| fig-cap: "Per-sample regression targets $\\epsilon$ (grey dots) against $x_t$ at $t = 300$; their bin averages (orange) lie on $-\\sigma_t\\,\\nabla_x \\log q_t(x)$ (blue), the marginal score scaled by the noise level."
#| fig-alt: "Scatter of grey dots filling a broad band from about -3 to 3 in epsilon over x_t from -3 to 3. Orange dots, the averages in narrow bins, follow a smooth blue curve that rises from about -2 to 2 overall, with a small dip between x_t = -1 and 0 where the two noised mixture components overlap."
fig, ax = plt.subplots(figsize=(7, 3))
ax.scatter(xt[:3000], eps[:3000], s=3, color="#c8c7c2", label="per-sample target $\\epsilon$")
ax.plot(xs, -sigma_t * score(xs, ab), color="#2a78d6", lw=2, label="$-\\sigma_t\\,s(x)$")
ax.plot(centers[well_filled], bin_mean[well_filled], "o", color="#eb6834", ms=4,
        label="bin mean of $\\epsilon$")
ax.set_xlim(-3, 3)
ax.set_ylim(-3.5, 3.5)
ax.set_xlabel("$x_t$")
ax.set_ylabel("noise")
ax.legend(fontsize=9, loc="upper left")
fig.tight_layout()
plt.show()

## The continuous view: diffusion as an SDE

[Song et al. (2021)](https://arxiv.org/abs/2011.13456) let $T \to \infty$. The DDPM chain becomes the **variance-preserving (VP) SDE** on $t \in [0, 1]$,

$$
dx = -\tfrac{1}{2}\beta(t)\,x\,dt + \sqrt{\beta(t)}\,dw,
$$

whose Euler–Maruyama step $x_{t+\Delta} = (1 - \tfrac12\beta\Delta)\,x_t + \sqrt{\beta\Delta}\,\epsilon \approx \sqrt{1-\beta_t}\,x_t + \sqrt{\beta_t}\,\epsilon$ is the DDPM forward step. (The NCSN of Song & Ermon is the *variance-exploding* SDE $dx = \sqrt{d\sigma^2(t)/dt}\;dw$.)

Anderson's theorem gives the generative process: the **reverse SDE**

$$
dx = \left[-\tfrac{1}{2}\beta(t)\,x - \beta(t)\,\nabla_x \log p_t(x)\right]dt + \sqrt{\beta(t)}\,d\bar w,
$$

run from $t = 1$ to $0$, with $s_\theta(x, t)$ plugged in for the score. Algorithm 2 is one discretization of it. Any numerical SDE solver, with any step size, is a sampler: the number of steps is a choice at inference, not a property of the model.


## The probability-flow ODE

The same paper's key result: for every SDE there is a **deterministic** ODE whose solution has the same marginals $p_t$ at every time,

$$
\boxed{\;dx = \left[f(x, t) - \tfrac{1}{2}\,g^2(t)\,\nabla_x \log p_t(x)\right]dt\;}
\qquad\text{(VP: } f = -\tfrac12\beta(t)\,x,\ g^2 = \beta(t)\text{)}.
$$

Half the score term, no noise. It is the reverse SDE with the stochastic part averaged out.

- **Same model, deterministic sampler.** Solve the ODE from $x_1 \sim \mathcal{N}(0, I)$ back to $t = 0$ with any ODE solver. DDIM ([Song, Meng & Ermon 2021](https://arxiv.org/abs/2010.02502)) is one such discretization, and ODE solvers tuned to this equation (DPM-Solver, Heun) give good images in 20–50 steps instead of 1000.
- **A bijection between noise and data.** Each noise vector maps to one image, so you can invert an image to its latent, interpolate, and edit.
- **Exact likelihoods**, through the change-of-variables formula for ODE flows (UDL chapter 16).
- **How few steps?** The error of an Euler step grows with the *curvature* of the trajectory. The VP ODE's paths are curved and most of their motion happens late. That observation leads to flow matching: if we are going to learn an ODE, why not learn one with straight paths?


## Flows: generating by solving an ODE

*Convention switch.* Flow matching papers run time from **noise at $t = 0$** to **data at $t = 1$**: $p_0 = \mathcal{N}(0, I)$, $p_1 = q$ (the data). Keep that in mind for the rest of the section.

A time-dependent **vector field** $u_t(x)$ defines a **flow** $\phi_t$ by the ODE $\frac{d}{dt}\phi_t(x) = u_t(\phi_t(x))$, $\phi_0(x) = x$. Pushing $p_0$ through $\phi_t$ gives a path of densities $p_t$; we say $u_t$ *generates* $p_t$. A continuous normalizing flow ([Chen et al. 2018](https://arxiv.org/abs/1806.07366)) is a network $v_\theta(x, t)$ playing the role of $u_t$, trained by maximum likelihood, which needs an ODE solve inside every training step and did not scale.

**Flow matching** ([Lipman et al., ICLR 2023](https://arxiv.org/abs/2210.02747)) replaces likelihood with regression. Pick *any* path $p_t$ from noise to data and a field $u_t$ that generates it, and train

$$
L_{\text{FM}}(\theta) = \mathbb{E}_{t \sim U[0,1],\ x \sim p_t}\left\|v_\theta(x, t) - u_t(x)\right\|^2 .
$$

Problem: we have samples of $p_1$ but no formula for a $p_t$ that connects it to $p_0$, let alone for the $u_t$ that generates it. The same obstacle as the marginal score.


## Conditional flow matching: straight paths per sample

Condition on one data point $x_1$ and the path becomes trivial. Draw $x_0 \sim \mathcal{N}(0, I)$, $x_1 \sim q$, and move in a **straight line**:

$$
x_t = (1-t)\,x_0 + t\,x_1, \qquad \frac{d x_t}{dt} = x_1 - x_0 .
$$

The conditional velocity is the constant $x_1 - x_0$, so the conditional target is free. The **conditional flow matching** loss is

$$
\boxed{\;L_{\text{CFM}}(\theta) = \mathbb{E}_{t,\ x_0 \sim \mathcal{N}(0,I),\ x_1 \sim q}\left\|v_\theta\!\big((1-t)x_0 + t x_1,\ t\big) - (x_1 - x_0)\right\|^2\;}
$$

Training is: sample a pair, pick a random $t$, interpolate, regress the difference. No schedule $\beta_t$, no $\bar\alpha_t$, no SDE.

## Conditional paths and the marginal field

In [ ]:
#| code-fold: true
#| fig-cap: "Left: a few conditional straight paths and their constant velocities. Right: the marginal field $u_t(x) = \\mathbb{E}[x_1 - x_0 \\mid x_t = x]$ that the regression converges to, with trajectories of the ODE it defines. Computed exactly for the 1D two-component Gaussian mixture."
#| fig-alt: "Two panels. Left: straight lines from noise samples at t=0 to data samples at t=1 in the (t, x) plane, with orange velocity arrows, crossing each other. Right: a grey arrow field of the marginal velocity in the (t, x) plane with nine blue ODE trajectories that fan out smoothly from the noise toward two data clusters without crossing."
# Flow-matching time: noise x_0 ~ N(0, 1) at t = 0, data x_1 at t = 1.
# Given data component k, x_t ~ N(t mu_k, (1-t)^2 + t^2 sd_k^2). A Gaussian path
# N(m(t), s(t)^2) is generated by u = m' + (s'/s)(x - m) (Lipman et al. 2023,
# Theorem 3); the marginal field averages it over the components' posterior.
def rf_velocity(x, t):
    m = t * MIX_MU
    s2 = (1 - t) ** 2 + t ** 2 * MIX_SD ** 2
    ds_over_s = (-(1 - t) + t * MIX_SD ** 2) / s2
    logp = np.log(MIX_W)[:, None] - 0.5 * np.log(s2)[:, None] - 0.5 * (x[None, :] - m[:, None]) ** 2 / s2[:, None]
    r = np.exp(logp - logp.max(axis=0))
    r /= r.sum(axis=0)
    u_k = MIX_MU[:, None] + ds_over_s[:, None] * (x[None, :] - m[:, None])
    return (r * u_k).sum(axis=0)

def rk4(f, x, t_grid):
    xs = [np.asarray(x, dtype=float)]
    for t0, t1 in zip(t_grid[:-1], t_grid[1:]):
        h, x = t1 - t0, xs[-1]
        k1 = f(x, t0)
        k2 = f(x + 0.5 * h * k1, t0 + 0.5 * h)
        k3 = f(x + 0.5 * h * k2, t0 + 0.5 * h)
        k4 = f(x + h * k3, t1)
        xs.append(x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4))
    return np.array(xs)

rng = np.random.default_rng(3)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.1))
n = 7
x0 = rng.normal(size=n)
x1 = sample_data(n, rng)
ts = np.linspace(0, 1, 50)
for a, b in zip(x0, x1):
    ax1.plot(ts, (1 - ts) * a + ts * b, color="#2a78d6", lw=1.4, alpha=0.85)
    ax1.annotate("", xy=(0.55, 0.45 * a + 0.55 * b), xytext=(0.4, 0.6 * a + 0.4 * b),
                 arrowprops=dict(arrowstyle="-|>", color="#eb6834", lw=1.6))
ax1.scatter(np.zeros(n), x0, color="#8a8984", zorder=3, s=28, label="$x_0 \\sim \\mathcal{N}(0,1)$")
ax1.scatter(np.ones(n), x1, color="#1d1d1b", zorder=3, s=28, label="$x_1 \\sim$ data")
ax1.set_title("(a) Conditional paths, velocity $x_1-x_0$ (arrows)", fontsize=11)
ax1.legend(loc="lower left", fontsize=9)

T_mesh, X_mesh = np.meshgrid(np.linspace(0.02, 0.98, 13), np.linspace(-3.3, 3.0, 15))
U = np.stack([rf_velocity(X_mesh[:, j], T_mesh[0, j]) for j in range(T_mesh.shape[1])], axis=1)
ax2.quiver(T_mesh, X_mesh, np.ones_like(U), U, angles="xy", scale_units="xy", scale=14,
           width=0.004, color="#8a8984")
t_grid = np.linspace(0, 1, 200)
trajectories = rk4(rf_velocity, np.linspace(-2.2, 2.2, 9), t_grid)
ax2.plot(t_grid, trajectories, color="#2a78d6", lw=1.6)
ax2.set_title("(b) Marginal field $u_t(x)$ and ODE trajectories", fontsize=11)
ax2.set_xlim(-0.03, 1.03)
for ax in (ax1, ax2):
    ax.set_xlabel("$t$")
    ax.set_ylabel("$x$")
    ax.set_ylim(-3.6, 3.2)
fig.tight_layout()
plt.show()

The per-sample targets on the left point in conflicting directions where paths cross; the network cannot fit them all, and the best it can do is their conditional average, the field on the right.

## Why the conditional loss has the same gradient

*Quizzable.* The marginal path $p_t(x) = \int p_t(x \mid x_1)\,q(x_1)\,dx_1$ is generated by the posterior-averaged field (Lipman et al., Theorem 1)

$$
u_t(x) = \int u_t(x \mid x_1)\,\frac{p_t(x \mid x_1)\,q(x_1)}{p_t(x)}\,dx_1 = \mathbb{E}\!\left[\,u_t(x_t \mid x_1)\ \middle|\ x_t = x\,\right].
$$

Expand both losses with $\|v - u\|^2 = \|v\|^2 - 2\,v\cdot u + \|u\|^2$ and compare term by term (Theorem 2):

- $\mathbb{E}\|v_\theta(x, t)\|^2$: identical in both, because sampling $x_1 \sim q$ then $x \sim p_t(\cdot \mid x_1)$ *is* sampling $x \sim p_t$.
- $\mathbb{E}\|u_t\|^2$ and $\mathbb{E}\|u_t(\cdot \mid x_1)\|^2$: different numbers, but neither depends on $\theta$.
- The cross term: $\displaystyle \mathbb{E}_{x \sim p_t}\big[v_\theta(x)\cdot u_t(x)\big] = \int v_\theta(x)\cdot\!\int u_t(x \mid x_1)\,p_t(x \mid x_1)\,q(x_1)\,dx_1\,dx = \mathbb{E}_{x_1 \sim q,\ x \sim p_t(\cdot \mid x_1)}\big[v_\theta(x)\cdot u_t(x \mid x_1)\big]$, after multiplying by $p_t(x)$ and swapping the integrals.
- Hence $L_{\text{FM}} = L_{\text{CFM}} + \text{const}$ and $\boxed{\nabla_\theta L_{\text{FM}} = \nabla_\theta L_{\text{CFM}}}$. The minimizer is $v^*(x, t) = \mathbb{E}[x_1 - x_0 \mid x_t = x]$.


This is the denoising-score-matching argument again, with a velocity in place of a score. The per-sample paths are straight; the marginal field, which the network actually learns, is their average and is not.


## Rectified flow: why straight paths mean few steps

[Liu, Gong & Liu (ICLR 2023)](https://arxiv.org/abs/2209.03003) arrived at the same objective (linear interpolation, independent $x_0, x_1$) from a transport perspective, and added the step that makes it fast.

- An ODE's trajectories cannot cross, so the learned flow *rewires* the crossing straight lines into non-crossing curves with the same endpoint distributions (the right panel of the previous figure).
- If a trajectory were exactly straight, **one Euler step would be exact**: $x_1 = x_0 + 1 \cdot v_\theta(x_0, 0)$. The Euler error on a curved path scales with curvature times step size, so straighter paths mean fewer steps for the same quality.
- **Reflow:** generate pairs $(x_0, \hat x_1)$ by running the learned ODE, then retrain on *those* pairs. The new pairs are already non-crossing, so the retrained flow's paths are straighter; repeat. Each round trades a little distribution fidelity for straighter paths, which is why one or two rounds are typical.
- Rectified flow is also exactly the diffusion model's probability-flow ODE on a different noise schedule ([Kingma & Gao 2023](https://arxiv.org/abs/2303.00848)); what is new is the choice of path and the explicit goal of straightness.


In [ ]:
#| code-fold: true
#| fig-cap: "Probability-flow ODE trajectories of the VP SDE (left, linear $\\beta(t)$ from 0.1 to 20) and rectified-flow ODE trajectories (right) from the same ten noise samples, with a 3-step Euler solve on two of them. Both computed exactly for the same 1D two-component Gaussian mixture."
#| fig-alt: "Two panels sharing a vertical axis x from -3 to 3. Left, DDPM: ten blue trajectories stay almost flat for the first half of the progress axis then bend sharply toward the two data modes at -2 and 1.5; two orange dashed 3-step Euler solves stay flat for two steps and make one large final jump, ending near -1.8 and 2.2 against exact endpoints of -2.0 and 2.0. Right, rectified flow: the same ten trajectories move from the start and reach the same endpoints along curved paths; the orange 3-step Euler solves end near -1.7 and 1.7."
# VP SDE dx = -1/2 beta(t) x dt + sqrt(beta(t)) dw on t in [0, 1], data at t = 0.
# beta(t) = 0.1 + 19.9 t is the continuous limit of the DDPM schedule (T * beta_t).
BETA_MIN, BETA_MAX = 0.1, 20.0

def beta_c(t):
    return BETA_MIN + t * (BETA_MAX - BETA_MIN)

def alpha_bar_c(t):
    return np.exp(-(BETA_MIN * t + 0.5 * (BETA_MAX - BETA_MIN) * t ** 2))

def vp_score(x, t):
    r, means, variances = responsibilities(x, alpha_bar_c(t))
    return (r * -(x[None, :] - means[:, None]) / variances[:, None]).sum(axis=0)

def vp_pf_ode_velocity(x, t):
    """dx/dt = f - 1/2 g^2 score, the probability-flow ODE of the VP SDE."""
    return -0.5 * beta_c(t) * x - 0.5 * beta_c(t) * vp_score(x, t)

def euler(f, x, t_grid):
    xs = [np.asarray(x, dtype=float)]
    for t0, t1 in zip(t_grid[:-1], t_grid[1:]):
        xs.append(xs[-1] + (t1 - t0) * f(xs[-1], t0))
    return np.array(xs)

noise = np.sort(np.random.default_rng(7).normal(size=10))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5.0, 2.6), sharey=True)
t_vp = np.linspace(1.0, 0.0, 400)                    # VP: noise at t = 1 -> data at t = 0
ax1.plot(1 - t_vp, rk4(vp_pf_ode_velocity, noise, t_vp), color="#2a78d6", lw=1.5)
t_rf = np.linspace(0.0, 1.0, 400)                    # RF: noise at t = 0 -> data at t = 1
ax2.plot(t_rf, rk4(rf_velocity, noise, t_rf), color="#2a78d6", lw=1.5)
coarse = np.linspace(0.0, 1.0, 4)
ax1.plot(coarse, euler(vp_pf_ode_velocity, noise[[0, 9]], 1 - coarse), "o--",
         color="#eb6834", lw=1.4, ms=4)
ax2.plot(coarse, euler(rf_velocity, noise[[0, 9]], coarse), "o--", color="#eb6834", lw=1.4, ms=4)
ax1.plot([], [], color="#2a78d6", label="exact ODE")
ax1.plot([], [], "o--", color="#eb6834", label="3 Euler steps")
ax1.legend(loc="upper left", fontsize=7)
ax1.set_title("DDPM: VP PF-ODE", fontsize=9)
ax2.set_title("Rectified flow", fontsize=9)
ax1.set_xlabel("noise → data", fontsize=9)
ax2.set_xlabel("$t$", fontsize=9)
ax1.set_ylabel("$x$")
ax1.set_ylim(-3.4, 3.0)
for ax in (ax1, ax2):
    for mu in MIX_MU:
        ax.axhline(mu, color="#8a8984", lw=0.8, ls=":")
    ax.tick_params(labelsize=8)
fig.tight_layout()
plt.show()

On this 1D toy the two ODEs reach the same endpoints, and *without reflow* neither path is straight: three Euler steps give a mean endpoint error of 0.35 for VP and 0.42 for rectified flow over 1,000 noise samples. The few-step advantage comes from reflow, which straightens the learned paths, and it matters most in high dimensions, where conditional paths cross far more than in 1D. Production rectified-flow models still sample with 20–50 steps; distilled ones use 1–4.


## One model, three languages

| | **DDPM** ([Ho et al. 2020](https://arxiv.org/abs/2006.11239)) | **Score SDE** ([Song et al. 2021](https://arxiv.org/abs/2011.13456)) | **Flow matching / rectified flow** ([Lipman 2023](https://arxiv.org/abs/2210.02747), [Liu 2023](https://arxiv.org/abs/2209.03003)) |
|---|---|---|---|
| network predicts | noise $\epsilon_\theta(x_t, t)$ | score $s_\theta(x, t) \approx \nabla_x \log p_t(x)$ | velocity $v_\theta(x, t) \approx \mathbb{E}[x_1 - x_0 \mid x_t]$ |
| path from data to noise | $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$ (VP) | VP or VE SDE marginals | $x_t = (1-t)\,x_0 + t\,x_1$ (noise at $t=0$), straight |
| training loss | $\|\epsilon - \epsilon_\theta\|^2$ | $\lambda(t)\,\|\nabla \log q(x_t \mid x_0) - s_\theta\|^2$ | $\|v_\theta - (x_1 - x_0)\|^2$ |
| sampler | ancestral, $T \approx 1000$ steps | reverse SDE, or PF-ODE (DDIM, DPM-Solver), 20–50 steps | Euler or Heun on the ODE, 20–50 steps; fewer after reflow or distillation |
| conversions | $\epsilon = -\sigma_t s$ | $s = -\epsilon/\sigma_t$ | on a Gaussian path $x_t = a_t x_1 + b_t \epsilon$: $v = \dot a_t\,\hat x_1 + \dot b_t\,\hat\epsilon$ |

All three networks are linear reparameterizations of the same conditional expectation of the data given $x_t$ ($\mathbb{E}[x_0 \mid x_t]$ in DDPM notation, $\mathbb{E}[x_1 \mid x_t]$ in the FM column). Changing the parameterization changes the implied loss weighting over $t$, not the optimum. The practical differences are the path (straight or not) and the weighting ([Kingma & Gao 2023](https://arxiv.org/abs/2303.00848)).


## In production: SD3 and Flux use rectified flow

[Esser et al. (ICML 2024)](https://arxiv.org/abs/2403.03206) compared diffusion and flow formulations at scale for Stable Diffusion 3 and chose rectified flow with two changes:

- **Logit-normal timestep sampling.** Instead of $t \sim U[0,1]$, sample $t$ from a logit-normal density that concentrates training on the middle of the path, where the velocity is hardest to predict (the ends are nearly trivial: pure noise or almost clean).
- **Resolution-dependent timestep shift.** Noising $m$ pixels instead of $n$ at the same $t$ destroys less information, because independent noise averages out over more pixels. SD3 (which writes the path with data at $t = 0$) shifts the timesteps so the signal-to-noise ratio matches a reference resolution (eq. 23):

  $$
  \begin{aligned}
  t_m &= \frac{\alpha\,t_n}{1 + \left(\alpha - 1\right)t_n}, \qquad \alpha = \sqrt{m/n} \;\; (= 4 \text{ at } 1024^2 \text{ against } 256^2); \\
  &\text{SD3 instead sets } \alpha = 3.0 \text{ at } 1024^2 \text{ by a human-preference study}.
  \end{aligned}
  $$

- Flux.1 ([Black Forest Labs, 2024](https://github.com/black-forest-labs/flux); open weights) uses the same recipe, as do most 2025–26 image and video models. The rectified-flow objective, a timestep shift and a transformer backbone are now the default; "diffusion model" in a 2026 paper usually means this.


The shift in SD3's convention, $x_t = (1-t)\,x_0 + t\,\epsilon$ with data at
$t = 0$: a shift above 1 moves every timestep toward the noise end, so the
sampler spends more of its steps at high noise. On this path the signal-to-noise ratio is
$\left((1-t)/t\right)^2$.

In [ ]:
#| code-fold: true
#| fig-cap: "The SD3 timestep shift $t_m = s\\,t_n / (1 + (s-1)\\,t_n)$ for shift $s = 1$ (none) and $s = 3$. Left: the shifted time. Right: the signal-to-noise ratio at each nominal time."
#| fig-alt: "Two panels against the nominal time t_n from 0 to 1. Left: shifted time t_m; the s = 1 line is the diagonal, the s = 3 curve bows above it, reaching 0.75 at t_n = 0.5. Right: log signal-to-noise ratio, decreasing from left to right; the s = 3 curve lies below the s = 1 curve by a constant amount, log of 9."
def shift_timestep(t, shift):
    return shift * t / (1 + (shift - 1) * t)

t_n = np.linspace(0.001, 0.999, 500)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3.2))
for shift, color in [(1.0, "#8a8984"), (3.0, "#2a78d6")]:
    t_m = shift_timestep(t_n, shift)
    ax1.plot(t_n, t_m, color=color, lw=1.8, label=f"shift $s = {shift:g}$")
    ax2.plot(t_n, np.log(((1 - t_m) / t_m) ** 2), color=color, lw=1.8, label=f"shift $s = {shift:g}$")
    print(f"shift {shift:g}: t_n = 0.5 -> t_m = {shift_timestep(0.5, shift):.2f}")
ax1.set_xlabel("nominal $t_n$")
ax1.set_ylabel("shifted $t_m$")
ax2.set_xlabel("nominal $t_n$")
ax2.set_ylabel("log SNR at $t_m$")
ax1.legend(fontsize=9)
fig.tight_layout()
plt.show()

The shift multiplies the noise-to-signal ratio $t/(1-t)$ by $s$
(check: $t_m/(1-t_m) = s\,t_n/(1-t_n)$), so in log SNR it is a constant offset
of $-2\log s$, and a shift of 3 lowers the SNR at every step by a factor of 9.


# Making it practical

A trained velocity or noise network is a long way from a text-to-image
product. This section covers the five pieces that close the gap: guidance (to
follow a prompt), latent diffusion (to afford high resolution), the diffusion
transformer (to scale), few-step distillation (to serve), and discrete
diffusion (to generate text at all).


## Classifier-free guidance

Conditional generation on a class or caption $c$ means learning $\epsilon_\theta(x_t, t, c)$; the conditioning is just another input. But a plain conditional model follows the prompt weakly. [Dhariwal & Nichol (2021)](https://arxiv.org/abs/2105.05233) sharpened it with a separate classifier's gradient; [Ho & Salimans (2022)](https://arxiv.org/abs/2207.12598) got the same effect with **no classifier**:

- **Train one network for both** $\epsilon_\theta(x_t, t, c)$ and $\epsilon_\theta(x_t, t, \varnothing)$, by replacing $c$ with a null token for 10–20% of training examples.
- **Sample with an extrapolated prediction**, with guidance weight $w$:

  $$
  \boxed{\;\tilde\epsilon_\theta(x_t, t, c) = (1 + w)\,\epsilon_\theta(x_t, t, c) - w\,\epsilon_\theta(x_t, t, \varnothing)\;}
  $$

- Using $\epsilon = -\sigma_t s$ and Bayes' rule, $\nabla_x \log p(c \mid x) = \nabla_x \log p(x \mid c) - \nabla_x \log p(x)$, so this is sampling with the score $\nabla_x \log p(x \mid c) + w\,\nabla_x \log p(c \mid x)$: the conditional score plus $w$ times an implicit classifier's gradient. The implied density is $\propto p(x \mid c)^{1+w}\,p(x)^{-w}$, a sharpened conditional, not a normalized model.
- **The CFG scale.** Libraries expose $s = 1 + w$ as `guidance_scale`, $\tilde\epsilon = \epsilon_\varnothing + s\,(\epsilon_c - \epsilon_\varnothing)$; $s = 1$ is the plain conditional model, typical values are 3–8. Larger $s$ improves prompt adherence and per-sample quality and reduces diversity; too large saturates colors and distorts. The same formula applies to the velocity $v_\theta$ in a flow model. Cost: two network evaluations per step.


## What guidance does to the distribution

In [ ]:
#| code-fold: true
#| fig-cap: "(a) The density classifier-free guidance implicitly samples from, $\\propto p(x \\mid c)^{1+w}\\,p(x)^{-w}$, for a 1D two-class example, as $w$ grows. (b) The scores being combined. Mass moves away from the region that the *other* class could also have produced and the distribution narrows: better adherence, less diversity."
#| fig-alt: "Two panels. Left: densities on x from -4 to 4.5. A grey dashed bimodal unconditional density, a dotted class-B density centred at 1.6, and four guided class-A densities for w = 0, 1, 3, 7 that become progressively taller, narrower and shifted left, with their right tails cut off near x = 0. Right: the conditional score (blue straight line), the unconditional score (grey dashed curve with a bump) and the guided score for w = 3 (orange), which matches the conditional score on the left and dives steeply negative to the right of x = -1."
# Two equally likely classes with overlapping Gaussian conditionals; guide toward A.
mu_a, mu_b, sd = -0.8, 1.6, 0.9
x = np.linspace(-5, 6, 1200)
dx = x[1] - x[0]
p_a, p_b = gauss(x, mu_a, sd ** 2), gauss(x, mu_b, sd ** 2)
p = 0.5 * p_a + 0.5 * p_b

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
ax1.plot(x, p, color="#8a8984", lw=1.6, ls="--", label="$p(x)$ (unconditional)")
ax1.plot(x, p_b, color="#8a8984", lw=1.0, ls=":", label="$p(x \\mid c=B)$")
for w, color in zip([0, 1, 3, 7], ["#2a78d6", "#1baf7a", "#eb6834", "#e34948"]):
    log_guided = (1 + w) * np.log(p_a) - w * np.log(p)
    guided = np.exp(log_guided - log_guided.max())
    guided /= guided.sum() * dx                    # normalize numerically
    ax1.plot(x, guided, color=color, lw=1.8, label=f"$w={w}$")
ax1.set_ylabel("density")
ax1.set_ylim(0, 0.85)
ax1.set_title("(a) CFG samples from $\\propto p(x\\mid c)^{1+w}\\, p(x)^{-w}$", fontsize=11)
ax1.legend(fontsize=8, loc="upper right", title="guided toward $c=A$", title_fontsize=8)

s_a = -(x - mu_a) / sd ** 2                        # score of p(x | A)
s_b = -(x - mu_b) / sd ** 2
r_a = 0.5 * p_a / p                                # p(A | x)
s_p = r_a * s_a + (1 - r_a) * s_b                  # score of p(x)
w = 3
ax2.plot(x, s_a, color="#2a78d6", lw=1.8, label="$\\nabla_x \\log p(x \\mid c)$")
ax2.plot(x, s_p, color="#8a8984", lw=1.6, ls="--", label="$\\nabla_x \\log p(x)$")
ax2.plot(x, (1 + w) * s_a - w * s_p, color="#eb6834", lw=1.8,
         label=f"guided, $w={w}$: $(1+w)\\,\\nabla \\log p(x\\mid c) - w\\,\\nabla \\log p(x)$")
ax2.axhline(0, color="#1d1d1b", lw=0.6)
ax2.set_ylim(-9, 7)
ax2.set_ylabel("score")
ax2.set_title("(b) Guidance adds $w\\,\\nabla_x \\log p(c \\mid x)$", fontsize=11)
ax2.legend(fontsize=8, loc="upper right")
for ax in (ax1, ax2):
    ax.set_xlim(-4, 4.5)
    ax.set_xlabel("$x$")
fig.tight_layout()
plt.show()

## Latent diffusion

![[Rombach et al., "High-Resolution Image Synthesis with Latent Diffusion Models" (arXiv Dec 2021, CVPR 2022)](https://arxiv.org/abs/2112.10752), Figure 3.](figs/18-Diffusion-and-Flow-Matching/latent-diffusion.png) to z, repeated T-1 times. A conditioning branch (semantic map, text, representations, images) is encoded by tau_theta and fed to the U-Net via cross-attention or concatenation through a switch." width="52%"}

- Train an autoencoder $(\mathcal{E}, \mathcal{D})$ first (KL- or VQ-regularized, as in lecture 17), downsampling by $f = 8$: a $512 \times 512 \times 3$ image becomes a $64 \times 64 \times 4$ latent, 48 times fewer numbers.
- Run the whole diffusion process in latent space; decode once at the end. Perceptually irrelevant high-frequency detail is handled by the autoencoder, and the diffusion model spends its capacity on semantics.
- Conditioning enters through **cross-attention** from the encoded prompt $\tau_\theta(y)$ into the U-Net. Stable Diffusion = this model with a CLIP text encoder, trained on LAION. Nearly every image and video generator since is latent.

## DiT: a transformer replaces the U-Net

![[Peebles & Xie, "Scalable Diffusion Models with Transformers" (ICCV 2023)](https://arxiv.org/abs/2212.09748), Figure 3.](figs/18-Diffusion-and-Flow-Matching/dit-architecture.png)

- **Patchify** the $32 \times 32 \times 4$ latent (a $256^2$ image through the Stable Diffusion VAE) into tokens, patch size $p \in \{2, 4, 8\}$: $p = 2$ gives 256 tokens. Then a standard ViT (lecture 16) with sinusoidal position embeddings.
- **adaLN-Zero:** the timestep and class embedding predict the scale and shift of every layer norm and a gate initialized to zero, so each block starts as the identity. It beat cross-attention and in-context conditioning in their ablations.
- Output: the noise $\epsilon$ (and a diagonal covariance). The model is a drop-in replacement for the U-Net in a latent diffusion pipeline.

## DiT scaling: FID follows compute

![Figure 8 of [Peebles & Xie (2023)](https://arxiv.org/abs/2212.09748): FID-50K on ImageNet $256^2$ after 400K training steps for twelve DiT models (sizes S/B/L/XL, patch sizes 8/4/2) against forward-pass Gflops.](figs/18-Diffusion-and-Flow-Matching/dit-gflops-vs-fid.png)


- FID improves monotonically with transformer Gflops, whether the compute comes from a **wider/deeper** model or **more tokens** (smaller patches). Correlation $-0.93$ on a log scale.
- Model size (S to XL, about 33M to 675M parameters) matters less than Gflops: XL/8 is worse than L/2.
- DiT-XL/2 reached FID 2.27 on class-conditional ImageNet $256^2$ with guidance, the best at the time, with far less compute than pixel-space U-Nets (ADM, 1,120 Gflops) and about the same as LDM-4 (119 vs 104 Gflops).
- The lesson the field took: the transformer scaling playbook (lecture 12) applies to generation. SD3, Flux, and the video models of lecture 19 are all DiTs.


## Conditioning on text: cross-attention or joint attention

- **Text encoders** are frozen pretrained models: CLIP text encoders (SD 1–2), T5-XXL ([Imagen](https://arxiv.org/abs/2205.11487); SD3 uses CLIP-G, CLIP-L and T5-XXL together), and by 2025 the text tower of a VLM or an LLM. Their token sequence is the conditioning $c$.
- **Cross-attention** (U-Nets, PixArt): image tokens query the text tokens; text is read-only.
- **MM-DiT joint attention** (SD3, Flux): text and image tokens are concatenated into *one* sequence with **separate weights per modality** but a shared attention, so both streams update each other at every block. Esser et al. found it beats cross-attention and a plain DiT at equal compute.


![Figure 2 of [Esser et al. (2024)](https://arxiv.org/abs/2403.03206): the Stable Diffusion 3 architecture (left) and one MM-DiT block (right). Three frozen text encoders feed a pooled vector $y$ (added to the timestep embedding) and a token sequence $c$; the image stream $x$ and text stream $c$ have separate weights and share one attention.](figs/18-Diffusion-and-Flow-Matching/sd3-mmdit.png)


## Few-step generation by distillation

Even on straight paths, 20–50 network evaluations with guidance is 40–100 forward passes per image. Three ways to get to one to four steps, all starting from a trained teacher:

- **Progressive distillation** ([Salimans & Ho, ICLR 2022](https://arxiv.org/abs/2202.00512)). A student learns to reproduce *two* DDIM steps of the teacher in one step; the student becomes the next teacher; repeat. Each round halves the step count; they went from 8192 steps to 4 on CIFAR-10 with little loss in FID.
- **Consistency models** ([Song, Dhariwal, Chen & Sutskever, ICML 2023](https://arxiv.org/abs/2303.01469)). Learn a single function $f_\theta(x_t, t)$ that maps *any* point on a PF-ODE trajectory to its clean endpoint, by enforcing $f_\theta(x_{t_{n+1}}, t_{n+1}) \approx f_{\theta^-}(x_{t_n}, t_n)$ for adjacent points on the same trajectory (with the boundary condition $f(x, 0) = x$). One step generates; a few steps refine. Can be distilled from a teacher or trained alone; latent consistency models bring it to Stable Diffusion.
- **Adversarial distillation** (ADD, [Sauer et al. 2023](https://arxiv.org/abs/2311.17042): SDXL-Turbo; LADD: SD3-Turbo). A discriminator judges the student's one-step samples against real images, alongside a distillation loss to the teacher. The GAN of lecture 17 survives here as a loss term.
- The price, in every case, is some diversity and some controllability (guidance is usually baked in at a fixed scale). Speed-quality-diversity is the three-way trade-off that samplers and distillation methods move around on.


## Discrete diffusion: text without autoregression

Autoregression (lectures 11–13) factorizes $p(x) = \prod_i p(x_i \mid x_{<i})$ and decodes one token at a time, left to right. **Masked discrete diffusion** replaces Gaussian noise with masking: the forward process masks each token independently with probability $t$; the reverse process predicts the masked tokens.

- **LLaDA** ([Nie et al. 2025](https://arxiv.org/abs/2502.09992)) trains an 8B bidirectional Transformer (no causal mask) from scratch on 2.3T tokens with

  $$
  \mathcal{L}(\theta) = -\,\mathbb{E}_{t,\,x_0,\,x_t}\left[\frac{1}{t}\sum_{i=1}^{L}\mathbf{1}[x_t^i = \mathrm{M}]\,\log p_\theta\!\left(x_0^i \mid x_t\right)\right],
  $$

  a cross-entropy on the masked positions, reweighted by $1/t$, which upper-bounds the negative log-likelihood. Same recipe as this lecture: a fixed corruption process, a learned reverse step, an ELBO.
- **Sampling** starts from a fully masked sequence and alternates predict-all and re-mask (keeping the confident tokens) for a fixed number of steps: parallel decoding, any-order infilling, no left-to-right constraint.
- LLaDA 8B was competitive with LLaMA 3 8B on their evaluations after the same SFT pipeline. **Gemini Diffusion** (Google DeepMind, 2025) and **Mercury** (Inception Labs, 2025) are production systems built on the idea, aimed at latency.
- Open: quality per unit of compute at scale, and how to get the KV-cache-style efficiency of autoregression. Watch this one.


# Applications

Image generation was the first application; the same model, conditioned on
something other than a caption, solves most image-to-image problems, and the
same framework now reaches audio, molecules and robot control.


## Super-resolution

- Downsample training images 4x and upsample with bicubic interpolation.
- Train a latent diffusion model, conditioned on the low-resolution input by concatenation, to recover the fine detail.
- The same idea underlies the cascades in Imagen and the upsamplers in the video models of lecture 19.

[Rombach et al. (2022)](https://arxiv.org/abs/2112.10752)


![ImageNet 64→256 super-resolution on ImageNet-Val. Rombach et al., Figure 9.](figs/18-Diffusion-and-Flow-Matching/super-resolution.png)


## Inpainting

- Mask part of the image and generate the masked region consistently with the rest.
- Don't change the unmasked part: at every step, re-impose the known pixels (or latents) and let the model fill only the hole.
- Easy to keep consistent because the latent code has spatial structure.

[Rombach et al. (2022)](https://arxiv.org/abs/2112.10752)


![Object removal with the inpainting model. Rombach et al., Figure 10.](figs/18-Diffusion-and-Flow-Matching/inpainting.png)


## Beyond images

The recipe (fixed corruption, learned reverse step, conditioning) transfers to any data with a continuous or maskable representation:

- **Audio.** Latent diffusion on spectrogram or waveform latents: AudioLDM ([Liu et al. 2023](https://arxiv.org/abs/2301.12503)), Stable Audio Open ([Evans et al. 2024](https://arxiv.org/abs/2407.14358), open weights).
- **Proteins.** RFdiffusion ([Watson et al., Nature 2023](https://doi.org/10.1038/s41586-023-06415-8)) fine-tunes a structure-prediction network on denoising to generate novel backbones for binder and enzyme design.
- **Molecular docking.** DiffDock ([Corso et al., ICLR 2023](https://arxiv.org/abs/2210.01776)) treats docking as diffusion over a ligand's pose (position, orientation, torsions) relative to a target protein: it **predicts binding poses** for a given molecule rather than generating new molecules.
- **Robot control.** Diffusion Policy ([Chi et al., RSS 2023](https://arxiv.org/abs/2303.04137)) generates action sequences conditioned on observations; the multimodal action distributions that regression averages away are exactly what a diffusion model represents well.
- **Video and world models:** lecture 19.

# Summary

## Summary

- A diffusion model has a **fixed forward process** (shrink and add Gaussian noise, $T$ small Markov steps) and a **learned reverse process**; the forward process turns any data distribution into $\mathcal{N}(0, I)$.
- The **diffusion kernel** $q(x_t \mid x_0) = \mathcal{N}(\sqrt{\bar\alpha_t}\,x_0, (1-\bar\alpha_t) I)$ lets us jump to any step in one line; it follows from composing Gaussian steps.
- The **ELBO** reduces to Gaussian KLs against the known posterior $q(x_{t-1} \mid x_t, x_0)$; predicting the noise and dropping the weights gives $L_{\text{simple}} = \mathbb{E}\|\epsilon - \epsilon_\theta(x_t, t)\|^2$, a simplified objective rather than a bound.
- Training is noise regression (Algorithm 1); sampling runs the chain backwards with fresh noise each step (Algorithm 2); the network is a time-conditioned U-Net.
- **Noise is score:** $\epsilon = -\sigma_t\,\nabla \log q(x_t \mid x_0)$, and regressing on the conditional score yields the marginal score, because least squares converges to the conditional mean.
- Diffusion is an **SDE**; its **probability-flow ODE** has the same marginals, is deterministic, and makes the step count a sampler choice (DDIM, DPM-Solver).
- **Flow matching** learns the ODE's velocity directly on straight per-sample paths $x_t = (1-t)x_0 + t x_1$ with target $x_1 - x_0$; the **conditional loss has the same gradient** as the marginal one. **Rectified flow** straightens paths so few steps suffice; SD3 and Flux use it with a timestep shift.
- Practical stack: **classifier-free guidance** $(1+w)\epsilon_c - w\epsilon_\varnothing$ for prompt adherence; **latent diffusion** for resolution; the **DiT** backbone, whose FID scales with Gflops; **distillation** (progressive, consistency, adversarial) for 1–4 steps; **masked discrete diffusion** for text.
- Next, lecture 19: the same models over time, for **video generation and world models**.

## For additional practice and reading

- UDL chapter 18 (diffusion) and chapter 16 (normalizing flows, for the ODE change-of-variables)
- UDL notebooks: 18.1 Diffusion Encoder in 1D ([Colab](https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap18/18_1_Diffusion_Encoder.ipynb)), 18.2 Training the Decoder ([Colab](https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap18/18_2_1D_Diffusion_Model.ipynb)), 18.3 Reparameterized Model ([Colab](https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap18/18_3_Reparameterized_Model.ipynb)), 18.4 Families of Diffusion Models, DDIM ([Colab](https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap18/18_4_Families_of_Diffusion_Models.ipynb))
- [Rocca, "Understanding Diffusion Probabilistic Models (DPMs)", 2022](https://medium.com/data-science/understanding-diffusion-probabilistic-models-dpms-1940329d6048), the source of the first half's figures
- [Lipman et al., "Flow Matching Guide and Code", 2024](https://arxiv.org/abs/2412.06264): the reference for the second half, with a PyTorch library
- [Holderrieth & Erives, "An Introduction to Flow Matching and Diffusion Models", MIT 6.S184 notes, 2025](https://arxiv.org/abs/2506.02070)
- Hugging Face [diffusion course](https://huggingface.co/learn/diffusion-course) and [Diffusers](https://github.com/huggingface/diffusers): run SD3.5 or Flux.1-schnell on one GPU and vary `guidance_scale` and `num_inference_steps`
- [CVPR 2023 tutorial, "Denoising Diffusion Models: A Generative Learning Big Bang"](https://cvpr.thecvf.com/virtual/2023/tutorial/18546)